# Day 26 — Economic Data Pipeline

## Objectives
- Retrieve multiple economic indicators from ALFRED
- Store historical vintage intervals in SQLite
- Validate downloaded observations
- Prepare reusable data for the investment dashboard

In [1]:

import os
import time
import sqlite3
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv(".env")
API_KEY = os.getenv("FRED_API_KEY")

if not API_KEY:
    raise ValueError("FRED_API_KEY is missing.")

conn = sqlite3.connect("hedge_fund.db")

SERIES = {
    "CPI": "CPIAUCSL",
    "UNEMPLOYMENT": "UNRATE",
    "REAL_GDP": "GDPC1",
    "FED_RATE": "FEDFUNDS"
}

def download_vintages(
    indicator,
    series_id,
    start="2024-01-01",
    end="2026-09-24"
):
    params = {
        "series_id": series_id,
        "api_key": API_KEY,
        "file_type": "json",
        "realtime_start": start,
        "realtime_end": end,
        "observation_start": start,
        "observation_end": end,
        "output_type": 1
    }

    response = requests.get(
        "https://api.stlouisfed.org/fred/series/observations",
        params=params,
        timeout=60
    )
    response.raise_for_status()

    data = pd.DataFrame(
        response.json()["observations"]
    )

    if data.empty:
        return pd.DataFrame()

    data = data.rename(columns={
        "date": "observation_date",
        "realtime_start": "available_date",
        "realtime_end": "vintage_end_date"
    })

    data["value"] = pd.to_numeric(
        data["value"],
        errors="coerce"
    )

    data = data.dropna(subset=["value"]).copy()
    data["indicator"] = indicator
    data["source"] = "ALFRED"

    return data[
        [
            "indicator",
            "observation_date",
            "available_date",
            "vintage_end_date",
            "value",
            "source"
        ]
    ]

In [2]:

all_data = []

for indicator, series_id in SERIES.items():
    print(f"Downloading {indicator}...")

    data = download_vintages(
        indicator,
        series_id
    )

    if not data.empty:
        all_data.append(data)
        print(f"Retrieved {len(data)} records.")
    else:
        print("No records returned.")

    time.sleep(1)

if not all_data:
    raise RuntimeError(
        "No economic data was downloaded."
    )

economic_pipeline_data = pd.concat(
    all_data,
    ignore_index=True
)

display(
    economic_pipeline_data.groupby(
        "indicator"
    ).agg(
        records=("value", "size"),
        first_observation=("observation_date", "min"),
        latest_observation=("observation_date", "max")
    )
)

Retrieved 66 records.
Retrieved 38 records.
Retrieved 34 records.
Retrieved 32 records.


,records,first_observation,latest_observation
indicator,,,
CPI,66,2024-01-01,2026-08-01
FED_RATE,32,2024-01-01,2026-08-01
REAL_GDP,34,2024-01-01,2026-04-01
UNEMPLOYMENT,38,2024-01-01,2026-08-01


In [3]:

columns = [
    "indicator",
    "observation_date",
    "available_date",
    "vintage_end_date",
    "value",
    "source"
]

conn.executemany("""
INSERT OR REPLACE INTO economic_vintages (
    indicator,
    observation_date,
    available_date,
    vintage_end_date,
    value,
    source
)
VALUES (?, ?, ?, ?, ?, ?)
""", economic_pipeline_data[
    columns
].itertuples(index=False, name=None))

conn.commit()

sql_summary = pd.read_sql_query("""
SELECT
    indicator,
    COUNT(*) AS records,
    MIN(observation_date) AS first_observation,
    MAX(observation_date) AS latest_observation,
    MIN(available_date) AS earliest_vintage
FROM economic_vintages
GROUP BY indicator
ORDER BY indicator
""", conn)

display(sql_summary)

,indicator,records,first_observation,latest_observation,earliest_vintage
0,CPI,66,2024-01-01,2026-08-01,2024-02-13
1,FED_RATE,32,2024-01-01,2026-08-01,2024-02-01
2,REAL_GDP,34,2024-01-01,2026-04-01,2024-04-25
3,UNEMPLOYMENT,38,2024-01-01,2026-08-01,2024-02-02


In [4]:

# Validate the imported economic data

quality = pd.read_sql_query("""
SELECT
    indicator,
    COUNT(*) AS records,
    COUNT(DISTINCT observation_date) AS observations,
    COUNT(DISTINCT available_date) AS vintage_dates,
    SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END) AS missing_values
FROM economic_vintages
GROUP BY indicator
ORDER BY indicator
""", conn)

display(quality)

expected = set(SERIES.keys())
actual = set(quality["indicator"])

assert expected.issubset(actual), "Missing economic indicators"
assert (quality["missing_values"] == 0).all(), "Missing values found"

invalid_dates = pd.read_sql_query("""
SELECT *
FROM economic_vintages
WHERE available_date < observation_date
""", conn)

assert invalid_dates.empty, (
    "Review records with availability before observation date"
)

print("PASS: Basic economic data validation completed.")

,indicator,records,observations,vintage_dates,missing_values
0,CPI,66,31,31,0
1,FED_RATE,32,32,32,0
2,REAL_GDP,34,10,28,0
3,UNEMPLOYMENT,38,31,31,0


PASS: Basic economic data validation completed.


In [5]:

from economic_pipeline import update_economic_database

update_economic_database()

CPI 31 observations downloaded
UNEMPLOYMENT 31 observations downloaded
REAL_GDP 10 observations downloaded
FED_RATE 32 observations downloaded
